# Ungraded Lab: Cats vs. Dogs Class Activation Maps (PyTorch)

You will again practice with CAMs in this lab and this time there will only be two classes: Cats and Dogs. You will be revisiting this exercise in this week's programming assignment so it's best if you become familiar with the steps discussed here, particularly in preprocessing the image and building the model.

> This notebook is a PyTorch port of the original TensorFlow/Keras lab. The `cats_vs_dogs` dataset from TensorFlow Datasets is replaced by the same Microsoft/Kaggle image archive read through a `torch.utils.data.Dataset`.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import cv2

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchinfo import summary

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

## Download and Prepare the Dataset

We will use the [Cats vs Dogs](https://www.microsoft.com/en-us/download/details.aspx?id=54765) dataset (the same images that TensorFlow Datasets serves as `cats_vs_dogs`). The images are labeled 0 for cats and 1 for dogs.

In [ ]:
import os
import urllib.request
import zipfile

data_url = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
data_file_name = "data/kagglecatsanddogs_5340.zip"
os.makedirs("data", exist_ok=True)

if not os.path.exists(data_file_name):
    print("downloading the dataset (about 800 MB)...")
    urllib.request.urlretrieve(data_url, data_file_name)
if not os.path.exists("data/catsdogs/PetImages"):
    with zipfile.ZipFile(data_file_name, 'r') as zip_ref:
        zip_ref.extractall("data/catsdogs/")
print("dataset ready")

In [ ]:
import platform
from PIL import Image, ImageFile

# DataLoader worker processes on macOS are started with "spawn", which cannot see classes defined
# inside a notebook (such as the Dataset below). "fork" works fine for the image decoding the workers do.
MP_CONTEXT = "fork" if platform.system() == "Darwin" else None

# a few images in this dataset are truncated; let PIL load what it can instead of raising
ImageFile.LOAD_TRUNCATED_IMAGES = True


def list_cats_vs_dogs_files(root="data/catsdogs/PetImages", cache="data/catsdogs/file_list.txt"):
    '''
    Returns a sorted list of (path, label) pairs, label 0 for cats and 1 for dogs.

    Files that cannot be decoded as images are skipped (like tfds does when it prepares the
    dataset). Checking all 25,000 files takes a minute, so the result is cached on disk.

    Args:
      root (string) -- folder holding the Cat and Dog subfolders
      cache (string) -- file the resulting list is cached in

    Returns:
      list -- (path, label) pairs, sorted and stable across runs
    '''
    if os.path.exists(cache):
        with open(cache) as f:
            return [(line.split("\t")[0], int(line.split("\t")[1])) for line in f.read().splitlines()]

    files = []
    for label, folder in enumerate(["Cat", "Dog"]):
        for name in sorted(os.listdir(os.path.join(root, folder))):
            path = os.path.join(root, folder, name)
            try:
                with Image.open(path) as img:
                    img.verify()
                files.append((path, label))
            except Exception:
                pass   # not a valid image
    with open(cache, "w") as f:
        f.write("\n".join(f"{p}\t{l}" for p, l in files))
    return files


def take_split(files, start, end):
    '''
    The equivalent of the tfds split syntax train[start:end], with the bounds as fractions.

    Args:
      files (list) -- the full list of (path, label) pairs
      start (float) -- fraction of the list to start at, 0.0 is the beginning
      end (float) -- fraction of the list to stop at, 1.0 is the end

    Returns:
      list -- the selected slice of (path, label) pairs
    '''
    n = len(files)
    return files[int(start * n):int(end * n)]


class CatsVsDogs(Dataset):
    '''yields (image, label) pairs; `transform` turns the PIL image into a tensor'''

    def __init__(self, files, transform):
        '''
        Stores the (path, label) pairs and the transform applied to each image.

        Args:
          files (list) -- (path, label) pairs, label 0 for cat and 1 for dog
          transform (callable) -- turns a PIL image into a tensor
        '''
        self.files = files
        self.transform = transform

    def __len__(self):
        '''
        Reports how many items this split holds.

        Returns:
          int -- number of images in this split
        '''
        return len(self.files)

    def __getitem__(self, idx):
        '''
        Loads image `idx`, forces it to RGB, and applies the transform.

        Args:
          idx (int) -- index of the image to fetch

        Returns:
          (tensor, int) -- the transformed image and its label
        '''
        path, label = self.files[idx]
        image = Image.open(path).convert("RGB")
        return self.transform(image), label

In [ ]:
files = list_cats_vs_dogs_files()
print(f"{len(files)} valid images")

train_files = take_split(files, 0.0, 0.8)        # train[:80%]
validation_files = take_split(files, 0.8, 0.9)   # train[80%:90%]
test_files = take_split(files, 0.9, 1.0)         # train[-10%:]

The cell below will preprocess the images and create batches before feeding it to our model.

In [ ]:
# resize to 300 x 300, convert to a float tensor and normalize the pixel values to [0, 1]
augment_images = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
])

# use the utility function above to preprocess the images
augmented_training_data = CatsVsDogs(train_files, augment_images)

# shuffle and create batches before training
train_batches = DataLoader(augmented_training_data, batch_size=32, shuffle=True, num_workers=4, multiprocessing_context=MP_CONTEXT, persistent_workers=True)

## Build the classifier

This will look familiar to you because it is almost identical to the previous model we built. The key difference is the output is just one unit. In Keras that unit was sigmoid activated; in PyTorch the sigmoid is folded into `nn.BCEWithLogitsLoss` and applied explicitly when we need a probability. This is because we're only dealing with two classes.

In [ ]:
model = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),     # GlobalAveragePooling2D
    nn.Flatten(),
    nn.Linear(128, 1),           # one output unit (logit)
).to(device)

summary(model, input_size=(1, 3, 300, 300), device=device)

The loss can be adjusted from last time to deal with just two classes. For that, we pick binary cross entropy.

In [ ]:
# Training will take a while to complete (25 epochs over 18,000 images). Time for a break!

loss_fn = nn.BCEWithLogitsLoss()
# alpha/eps are set to the Keras RMSprop defaults (rho=0.9, epsilon=1e-7); PyTorch's defaults (0.99, 1e-8) take much larger first steps
optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001, alpha=0.9, eps=1e-7)

EPOCHS = 25
for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, count = 0.0, 0, 0
    for images, labels in train_batches:
        images, labels = images.to(device), labels.to(device).float().unsqueeze(1)
        logits = model(images)
        loss = loss_fn(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(images)
        correct += ((logits > 0) == (labels > 0.5)).sum().item()
        count += len(images)
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {total_loss / count:.4f} - accuracy: {correct / count:.4f}")

## Building the CAM model

You will follow the same steps as before in generating the class activation maps.

In [ ]:
# weights of the last linear layer, transposed to (features, units) = (128, 1) like Keras stores them
gap_weights = model[-1].weight.detach().cpu().numpy().T
print(gap_weights.shape)


class CAMModel(nn.Module):
    '''same model, but also returns the output of the last convolution layer'''

    def __init__(self, model):
        '''
        Splits the trained classifier into a feature half and a classifier half.

        Args:
          model (nn.Sequential) -- the trained cats vs dogs classifier
        '''
        super().__init__()
        self.features = model[:11]      # up to and including the last conv + ReLU -> (N, 128, 37, 37)
        self.classifier = model[11:]    # global average pooling -> flatten -> linear

    def forward(self, x):
        '''
        Runs the image through both halves of the split model.

        Args:
          x (tensor) -- batch of images, shape (N, 3, 300, 300)

        Returns:
          (tensor, tensor) -- last conv features (N, 128, 37, 37) and the sigmoid output (N, 1)
        '''
        features = self.features(x)
        logits = self.classifier(features)
        return features, torch.sigmoid(logits)


cam_model = CAMModel(model).to(device).eval()
print(cam_model)


def cam_predict(tensor_image, cam_model, device):
    '''
    Runs the CAM model on one image.

    Args:
      tensor_image (tensor) -- preprocessed image, shape (1, 3, 300, 300)
      cam_model (nn.Module) -- model returning (features, sigmoid output)
      device (torch.device) -- device to run on

    Returns:
      (array, array) -- features in channels-last layout (1, 37, 37, 128), and sigmoid outputs (1, 1)
    '''
    with torch.no_grad():
        features, results = cam_model(tensor_image.to(device))
    return features.permute(0, 2, 3, 1).cpu().numpy(), results.cpu().numpy()

In [ ]:
def show_cam(image_value, features, results, gap_weights):
  '''
  Displays the class activation map of an image

  Args:
    image_value (tensor) -- preprocessed input image with size 300 x 300, shape (1, 3, 300, 300)
    features (array) -- features of the image, shape (1, 37, 37, 128)
    results (array) -- output of the sigmoid layer
    gap_weights (array) -- weights of the final Linear layer, shape (128, 1)
  '''

  # there is only one image in the batch so we index at `0`
  features_for_img = features[0]
  prediction = results[0]

  # there is only one unit in the output so we get the weights connected to it
  class_activation_weights = gap_weights[:,0]

  # upsample to the image size
  class_activation_features = sp.ndimage.zoom(features_for_img, (300/37, 300/37, 1), order=2)

  # compute the intensity of each feature in the CAM
  cam_output  = np.dot(class_activation_features,class_activation_weights)

  # visualize the results
  print(f'sigmoid output: {results}')
  print(f"prediction: {'dog' if round(float(results[0][0])) else 'cat'}")
  plt.figure(figsize=(8,8))
  plt.imshow(cam_output, cmap='jet', alpha=0.5)
  # the image tensor is (1, 3, 300, 300); matplotlib wants (300, 300, 3)
  plt.imshow(image_value[0].permute(1, 2, 0), alpha=0.5)
  plt.show()

## Testing the Model

Let's download a few images and see how the class activation maps look like.

In [ ]:
test_images = {
    "cat1.jpg": "https://storage.googleapis.com/tensorflow-1-public/tensorflow-3-temp/MLColabImages/cat1.jpeg",
    "cat2.jpg": "https://storage.googleapis.com/tensorflow-1-public/tensorflow-3-temp/MLColabImages/cat2.jpeg",
    "catanddog.jpg": "https://storage.googleapis.com/tensorflow-1-public/tensorflow-3-temp/MLColabImages/catanddog.jpeg",
    "dog1.jpg": "https://storage.googleapis.com/tensorflow-1-public/tensorflow-3-temp/MLColabImages/dog1.jpeg",
    "dog2.jpg": "https://storage.googleapis.com/tensorflow-1-public/tensorflow-3-temp/MLColabImages/dog2.jpeg",
}
for name, url in test_images.items():
    if not os.path.exists(os.path.join("data", name)):
        urllib.request.urlretrieve(url, os.path.join("data", name))

In [ ]:
# utility function to preprocess an image and show the CAM
def convert_and_classify(image, cam_model, gap_weights, device):
  '''
  Loads an image file, preprocesses it like the training data, and shows its CAM.

  Args:
    image (string) -- path to a JPEG file
    cam_model (nn.Module) -- model returning (features, sigmoid output)
    gap_weights (array) -- weights of the final Linear layer, shape (128, 1)
    device (torch.device) -- device the model runs on
  '''
  # load the image (OpenCV loads BGR, so convert to the RGB channel order the model was trained on)
  img = cv2.imread(image)
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

  # preprocess the image before feeding it to the model
  img = cv2.resize(img, (300,300)) / 255.0

  # add a batch dimension because the model expects it, and move the channels first: (1, 3, 300, 300)
  tensor_image = torch.from_numpy(img).float().permute(2, 0, 1).unsqueeze(0)

  # get the features and prediction
  features,results = cam_predict(tensor_image, cam_model, device)

  # generate the CAM
  show_cam(tensor_image, features, results, gap_weights)

convert_and_classify('data/cat1.jpg', cam_model, gap_weights, device)
convert_and_classify('data/cat2.jpg', cam_model, gap_weights, device)
convert_and_classify('data/catanddog.jpg', cam_model, gap_weights, device)
convert_and_classify('data/dog1.jpg', cam_model, gap_weights, device)
convert_and_classify('data/dog2.jpg', cam_model, gap_weights, device)

Let's also try it with some of the test images before we make some observations.

In [ ]:
# preprocess the test images
augmented_test_data = CatsVsDogs(test_files, augment_images)
test_batches = DataLoader(augmented_test_data, batch_size=1)


for i, (img, lbl) in enumerate(test_batches):
  if i == 5:
    break
  print(f"ground truth: {'dog' if lbl.item() else 'cat'}")
  features,results = cam_predict(img, cam_model, device)
  show_cam(img, features, results, gap_weights)

If your training reached 80% accuracy, you may notice from the images above that the presence of eyes and nose play a big part in determining a dog, while whiskers and a colar mostly point to a cat. Some can be misclassified based on the presence or absence of these features. This tells us that the model is not yet performing optimally and we need to tweak our process (e.g. add more data, train longer, use a different model, etc).